# Analysis for the Inhalation POD Manuscript

This notebook uses persisted model results and the repository's analysis functions; it does not train or refit models.

In [1]:
import os
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
repo_root = next(
    (
        path
        for path in (working_directory, *working_directory.parents)
        if (path / 'config_inhalation.json').is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        'Could not locate the repository root from the working directory.'
    )
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config_management import UnifiedConfiguration
from data_management import DataManager
from results_analysis import ResultsAnalyzer
from results_management import ResultsManager
from vapor_concentration import (
    vapor_concentration_interval_classification_table,
)

config = UnifiedConfiguration('config_inhalation.json')
data_manager = DataManager(config.data, config.path)
results_manager = ResultsManager(
    output_dir=config.path.results_dir,
    results_file_type=config.data.file_type,
)
results_analyzer = ResultsAnalyzer(
    results_manager,
    data_manager,
    config.plot,
)

## Sensitivity Analysis

In [2]:
table_1 = results_analyzer.model_performance_table()
table_1.to_html(
    'Analyses/Inhalation/Generated/table_1_inhalation_sensitivity.html',
    index=False,
)
table_1

,QSAR Model (n),RMSE,MedAE,R²
0,General non-cancer effects,,,
1,RandomForestRegressor with feature selection (...,0.88 [0.76–1.03],0.50 [0.42–0.60],0.51 [0.39–0.63]
2,*RandomForestRegressor (415),0.87 [0.74–1.03],0.50 [0.42–0.59],0.52 [0.41–0.63]
3,*GradientBoostingRegressor (415),0.89 [0.75–1.05],0.51 [0.41–0.63],0.51 [0.36–0.64]
4,*Ridge (415),0.95 [0.82–1.12],0.57 [0.48–0.69],0.43 [0.24–0.58]
5,*LinearRegression (415),0.98 [0.83–1.13],0.58 [0.47–0.68],0.41 [0.21–0.58]
6,*XGBRegressor (415),0.93 [0.82–1.06],0.53 [0.43–0.65],0.46 [0.30–0.59]
7,*SVR (415),1.17 [1.05–1.33],0.76 [0.65–0.90],0.14 [0.04–0.25]
8,*MLPRegressor (415),1.03 [0.89–1.17],0.62 [0.51–0.74],0.35 [0.13–0.49]
9,**RDKit Features (415),0.91 [0.79–1.06],0.51 [0.41–0.67],0.48 [0.37–0.58]


## Vapor Concentration Ceiling

The calculation uses processed OPERA `VP_pred` in linear mmHg and `MolWeight` in g/mol, together with persisted endpoint BMCh medians and 90% prediction intervals. SVC is calculated at 298.15 K with vapor pressure capped at 760 mmHg. Endpoint summaries exclude the corresponding fitted training chemicals; missing or invalid physical-property inputs remain unclassified.

In [3]:
vapor_concentration_table = (
    vapor_concentration_interval_classification_table(
        data_manager,
        config.path,
        label_for_effect=config.plot.label_for_effect,
    )
)
vapor_concentration_table.to_html(
    'Analyses/Inhalation/Generated/vapor_concentration_summary.html',
    index=False,
)
vapor_concentration_table

,90% BMCh prediction-interval classification,"General noncancer, n (%)","Reproductive/developmental, n (%)"
0,Entirely at or below SVC,"75,283 (9.84)","40,113 (5.24)"
1,Crosses SVC,"234,550 (30.67)","190,631 (24.92)"
2,Entirely above SVC,"454,905 (59.49)","534,275 (69.84)"


## Cross-route Prediction Comparison

Summarize oral and inhalation QSAR predictions and Aurisano PODs for each endpoint.


In [1]:
from cross_route import (
    load_cross_route_comparison,
    summarize_cross_route,
    cross_route_summary_table,
)

oral_config = UnifiedConfiguration(
    config.plot.cross_route_config_files['oral'],
)
(cross_route_tables, cross_route_source_tables,
 cross_route_audit) = load_cross_route_comparison(
    oral_config, config,
)
cross_route_summary = summarize_cross_route(
    cross_route_tables, cross_route_source_tables,
)
cross_route_audit.set_index('effect').T


effect,general,repro_dev
oral_prediction_rows,803494,803494
inhalation_prediction_rows,803494,803494
duplicate_prediction_ids,0,0
oral_only_ids,0,0
inhalation_only_ids,0,0
matched_ids,803494,803494
valid_pairs,803494,803494
invalid_pairs,0,0
oral_source_ids,8023,6697
inhalation_source_ids,2095,638


In [2]:
import pandas as pd
from IPython.display import display

cross_route_table = cross_route_summary_table(
    cross_route_summary, config.plot.label_for_effect,
)
cross_route_table.to_html(
    'Analyses/Inhalation/Generated/cross_route_summary.html',
    index=False,
)
with pd.option_context('display.max_colwidth', None):
    display(cross_route_table)


,Statistic,General Noncancer: QSAR predictions,General Noncancer: Aurisano PODs,Reproductive/Developmental: QSAR predictions,Reproductive/Developmental: Aurisano PODs
0,"Paired PODs, n","803,494","1,460","803,494",313
1,"Oral POD, mg/(kg day), median (5th-95th percentile)",8.05 (1.8 to 63.3),23.4 (0.479 to 382),24.5 (4.62 to 136),67.7 (1.21 to 453)
2,"Inhalation administered dose, mg/(kg day), median (5th-95th percentile)",1.58 (0.422 to 33.3),25 (0.064 to 1.43e+03),4.27 (1.23 to 119),162 (0.503 to 2.62e+03)
3,"log10(oral/inhalation dose), median (5th-95th percentile)",0.648 (-0.504 to 1.73),0.0633 (-2.37 to 2.62),0.633 (-0.718 to 1.61),-0.072 (-1.82 to 1.71)
4,Spearman rho,0.075,0.211,0.221,0.477
5,"Oral POD >10 times inhalation dose, n (%)","240,859 (29.98%)",382 (26.16%),"233,940 (29.12%)",45 (14.38%)
6,"Inhalation dose >10 times oral POD, n (%)","10,493 (1.31%)",285 (19.52%),"20,567 (2.56%)",59 (18.85%)
7,">10-fold divergence in either direction, n (%)","251,352 (31.28%)",667 (45.68%),"254,507 (31.68%)",104 (33.23%)
